In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_root_dir = os.path.join(path, "PlantVillage/train")
train_dataset = ImageFolder(root=train_root_dir, transform=transform_train)

test_root_dir = os.path.join(path, "PlantVillage/test")
test_dataset = ImageFolder(root=test_root_dir, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
train_dataset[1700][1]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

idx_class = {0:"Early_blight",1:"Late_blight",2:"healthy"}

# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [1000,233,1500,1700,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].set_title(f'Class: {idx_class[label]}')

In [ ]:
# Write your code here
import torch.nn as nn
import torch.optim as optim
import torch
class CNN5layers(nn.Module):
    def __init__(self,input_channels=3,output_channels=3):
        super(CNN5layers, self).__init__()

        # Convolutional Layers
        self.classifier = nn.Sequential(
        nn.Conv2d(in_channels=input_channels, out_channels=16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
        nn.Flatten(),
        nn.Linear(32*32*64,128),
        nn.ReLU(inplace=True),
        nn.Linear(128,output_channels)

        )
    def forward(self, x):
        x = self.classifier(x)
        return x


In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy



In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN5layers(3,3).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")



In [ ]:
# Write your code here
class CNN5layersRes(nn.Module):
    def __init__(self,input_channels=3,output_channels=3):
        super(CNN5layersRes, self).__init__()

        # Convolutional Layers

        self.cnn1 = nn.Conv2d(in_channels=input_channels, out_channels=16, kernel_size=3, padding=1)
        self.bnf16 = nn.BatchNorm2d(16)
        self.bns16 = nn.BatchNorm2d(16)

        self.reul = nn.ReLU(inplace=True)
        self.cnn2 = nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, padding=1)
        self.cnn3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn32 = nn.BatchNorm2d(32)
        self.cnn4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn64 = nn.BatchNorm2d(64)
        self.cnn5 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.flatten = nn.Flatten(),
        self.fc1 = nn.Linear(32*32*64,128)
        self.fc2 = nn.Linear(128,output_channels)


    def forward(self, x):
        x1 = self.cnn1(x)
        x2 = self.bnf16(x1)
        x3 = self.relu(x2)

        x4 = self.cnn2(x3)
        x5 = self.bns16(x4)
        x6 = self.relu(x5)

        x7 = self.cnn3(x6)
        x8 = self.bn32(x7)
        x9 = self.relu(x8)

        x10 = self.cnn4(x9 + x4)
        x11 = self.bn64(x10)
        x12 = self.relu(x11)

        x13 = self.cnn5(x12)
        x14 = self.flatten(x13)

        x15 = self.fc1(x14)
        x16 = self.fc2(x15)
        return x16

In [ ]:
CNN5layersRes()